# 🎬 ClipForge — Local AI Video Clipper (Google Colab Edition)

Turn any YouTube video URL or uploaded file into short, reframed, styled-captioned clips directly inside Google Colab!

### ✨ Features
- **Zero API Keys Required**: 100% self-contained local processing with `faster-whisper` and `yt-dlp`.
- **Production-Grade Downloader Architecture**: Programmatic Netscape cookie parser & validator, persistent `/content/cookies.txt` copy isolation, automatic yt-dlp upgrades, 7-client rotation (`android`, `tv`, `ios`, `web`, `mweb`, `embedded`, `music`), User-Agent & HTTP header spoofing, network retry logic, and zero-crash error shielding.
- **Intelligent Virality Selector**: Automatically finds high-density speech windows and hook statements.
- **Creator Caption Styles**: Burn styled ASS subtitles (Hormozi, MrBeast, Bold White, Karaoke Yellow, Boxed TikTok, etc.).
- **Smart Reframing**: Professional 9:16 vertical Shorts/Reels layout with blurred background, sharp uncropped foreground, and dynamic caption positioning.
- **Google Colab Optimized**: Automatic CUDA GPU acceleration / CPU fallback & optional Google Drive output export.


In [ ]:
# ==========================================
# 1. DEPENDENCY INSTALLATION & HARDWARE PROBE
# ==========================================
import os
import sys
import subprocess

print("📦 Upgrading yt-dlp & installing dependencies...")
packages = [
    "yt-dlp",
    "faster-whisper",
    "indic-transliteration",
    "gradio",
    "requests",
    "pydantic>=2.0",
]

# Force upgrade yt-dlp to latest release
try:
    subprocess.run([sys.executable, "-m", "pip", "install", "-U", "--upgrade", "yt-dlp"], check=True)
    print("✅ yt-dlp updated to latest version.")
except Exception as e:
    print(f"⚠️ Warning updating yt-dlp: {e}")

for pkg in packages:
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", pkg], check=True)
    except Exception as e:
        print(f"⚠️ Warning installing {pkg}: {e}")

# Verify FFmpeg
try:
    ffmpeg_ver = subprocess.run(["ffmpeg", "-version"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    if ffmpeg_ver.returncode == 0:
        print("✅ FFmpeg is installed and ready.")
    else:
        print("⚠️ Installing FFmpeg...")
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=True)
except Exception:
    print("⚠️ Installing FFmpeg via apt...")
    subprocess.run(["apt-get", "update", "-qq"])
    subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"])

# Probe CUDA / GPU availability
cuda_available = False
gpu_name = "CPU"
try:
    import torch
    cuda_available = torch.cuda.is_available()
    if cuda_available:
        gpu_name = torch.cuda.get_device_name(0)
except Exception:
    try:
        from ctranslate2 import get_cuda_device_count
        cuda_available = get_cuda_device_count() > 0
        if cuda_available:
            gpu_name = "NVIDIA CUDA GPU"
    except Exception:
        cuda_available = False

print(f"🚀 Compute Hardware Detected: {gpu_name} (CUDA Acceleration: {cuda_available})")


## 📁 Section 2: Directory Layout & Google Drive Integration


In [ ]:
# ==========================================
# 2. SYSTEM PATHS & DIRECTORY SETUP
# ==========================================
import os
import platform
from pathlib import Path

# Smart path resolution for Google Colab vs Local PC
if os.path.exists("/content") or platform.system() == "Linux":
    BASE_DIR = Path("/content")
else:
    BASE_DIR = Path.cwd() / "content"

DOWNLOADS_DIR = BASE_DIR / "downloads"
TRANSCRIPTS_DIR = BASE_DIR / "transcripts"
CLIPS_DIR = BASE_DIR / "clips"
ASSETS_DIR = BASE_DIR / "assets"
FONTS_DIR = ASSETS_DIR / "fonts"
MASKS_DIR = ASSETS_DIR / "masks"
MUSIC_DIR = ASSETS_DIR / "music"
EXPORT_DIR = BASE_DIR / "exports"
PERSISTENT_COOKIE_FILE = BASE_DIR / "cookies.txt"

for d in [DOWNLOADS_DIR, TRANSCRIPTS_DIR, CLIPS_DIR, FONTS_DIR, MASKS_DIR, MUSIC_DIR, EXPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"📁 Local directories initialized successfully under {BASE_DIR}")

def mount_gdrive():
    """Mount Google Drive for persistent shorts export."""
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        drive_path = Path('/content/drive/MyDrive/ClipForge_Shorts')
        drive_path.mkdir(parents=True, exist_ok=True)
        print(f"✅ Google Drive mounted! Export directory: {drive_path}")
        return drive_path
    except Exception as e:
        print(f"ℹ️ Google Drive mount skipped or unavailable: {e}")
        return None


## 🔤 Section 3: Fonts & Rounded Mask Initialization


In [ ]:
# ==========================================
# 3. ASSETS & FONT DOWNLOADER
# ==========================================
import urllib.request

_GF = "https://raw.githubusercontent.com/google/fonts/main"

_CORE_FONTS = {
    "Roboto-Regular.ttf": "https://github.com/googlefonts/roboto-2/raw/main/src/hinted/Roboto-Regular.ttf",
    "Roboto-Bold.ttf": "https://github.com/googlefonts/roboto-2/raw/main/src/hinted/Roboto-Bold.ttf",
}

TRENDING_FONTS = {
    "Anton": ("Anton-Regular.ttf", f"{_GF}/ofl/anton/Anton-Regular.ttf"),
    "Bebas Neue": ("BebasNeue-Regular.ttf", f"{_GF}/ofl/bebasneue/BebasNeue-Regular.ttf"),
    "Poppins": ("Poppins-Bold.ttf", f"{_GF}/ofl/poppins/Poppins-Bold.ttf"),
    "Montserrat": ("Montserrat.ttf", f"{_GF}/ofl/montserrat/Montserrat%5Bwght%5D.ttf"),
    "Oswald": ("Oswald.ttf", f"{_GF}/ofl/oswald/Oswald%5Bwght%5D.ttf"),
    "Bangers": ("Bangers-Regular.ttf", f"{_GF}/ofl/bangers/Bangers-Regular.ttf"),
    "Luckiest Guy": ("LuckiestGuy-Regular.ttf", f"{_GF}/apache/luckiestguy/LuckiestGuy-Regular.ttf"),
    "Permanent Marker": ("PermanentMarker-Regular.ttf", f"{_GF}/apache/permanentmarker/PermanentMarker-Regular.ttf"),
    "Alfa Slab One": ("AlfaSlabOne-Regular.ttf", f"{_GF}/ofl/alfaslabone/AlfaSlabOne-Regular.ttf"),
    "DM Serif Display": ("DMSerifDisplay-Regular.ttf", f"{_GF}/ofl/dmserifdisplay/DMSerifDisplay-Regular.ttf"),
}

def download_asset_file(url: str, dest: Path) -> bool:
    if dest.exists() and dest.stat().st_size > 0:
        return True
    try:
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req) as resp, open(dest, 'wb') as f:
            f.write(resp.read())
        return True
    except Exception as e:
        print(f"⚠️ Warning downloading font {dest.name}: {e}")
        return False

def ensure_fonts():
    print("🔤 Downloading core & trending creator fonts...")
    for filename, url in _CORE_FONTS.items():
        download_asset_file(url, FONTS_DIR / filename)
    for family, (fname, url) in TRENDING_FONTS.items():
        download_asset_file(url, FONTS_DIR / fname)

ensure_fonts()

def ensure_rounded_mask(size: int = 1020, radius: int = 60) -> Path:
    path = MASKS_DIR / f"rounded_{size}_{radius}.png"
    if path.exists() and path.stat().st_size > 0:
        return path
    MASKS_DIR.mkdir(parents=True, exist_ok=True)
    
    if radius <= 0:
        cmd = ["ffmpeg", "-y", "-f", "lavfi", "-i", f"color=c=white:s={size}x{size}:d=0.1", "-frames:v", "1", str(path)]
    else:
        edge = size - 1 - radius
        expr = f"255*clip(0.5+({radius}-hypot(max(0\\,{radius}-X)+max(0\\,X-{edge})\\,max(0\\,{radius}-Y)+max(0\\,Y-{edge})))/1.5\\,0\\,1)"
        cmd = ["ffmpeg", "-y", "-f", "lavfi", "-i", f"color=c=black:s={size}x{size}:d=0.1", "-vf", f"geq=lum='{expr}':cb=128:cr=128", "-frames:v", "1", str(path)]
    
    try:
        subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True)
    except Exception as e:
        print(f"⚠️ Mask generation warning: {e}")
    return path

ensure_rounded_mask()
print("🎨 Rounded-corner video mask initialized.")


## 🎨 Section 4: Data Models & Creator Caption Presets


In [ ]:
# ==========================================
# 4. DATA MODELS & CAPTION PRESETS
# ==========================================
from enum import Enum
from typing import Optional, List, Dict, Any
from pydantic import BaseModel, Field

class AspectRatio(str, Enum):
    NINE_16 = "9:16"
    SIXTEEN_9 = "16:9"

class FitMode(str, Enum):
    CROP = "crop"
    SQUARE = "square"

class Device(str, Enum):
    AUTO = "auto"
    CUDA = "cuda"
    CPU = "cpu"

_BASE_PRESET = {
    "font_family": "Roboto",
    "bold": True,
    "font_size": 90,
    "primary_color": "#FFFFFF",
    "highlight_color": "#FFD400",
    "outline_color": "#000000",
    "outline": 5,
    "shadow": 1,
    "position": "bottom",
    "karaoke": False,
    "uppercase": True,
    "animation": "none",
    "max_lines": 2,
    "max_chars": 22,
    "background_enabled": False,
    "background_color": "#000000",
}

def _P(label: str, **kw) -> dict:
    d = dict(_BASE_PRESET)
    d["label"] = label
    d.update(kw)
    return d

STYLE_PRESETS: Dict[str, dict] = {
    "bold_white": _P("Bold White", highlight_color="#FFFFFF", font_size=96, max_chars=20),
    "karaoke_yellow": _P("Karaoke Yellow", karaoke=True, font_size=92, highlight_color="#FFE600"),
    "minimal": _P("Minimal", bold=False, uppercase=False, font_size=64, outline=1, shadow=2, highlight_color="#FFFFFF", max_chars=28),
    "hormozi_green": _P("Hormozi Green", font_family="Montserrat", animation="highlight", highlight_color="#27E36B", font_size=94, outline=6, position="center", max_lines=2, max_chars=16),
    "hormozi_yellow": _P("Hormozi Yellow", font_family="Montserrat", animation="highlight", highlight_color="#FFD400", font_size=94, outline=6, position="center", max_lines=2, max_chars=16),
    "beast_red": _P("Beast Pop", font_family="Anton", bold=False, animation="highlight", highlight_color="#FF3B30", font_size=108, outline=7, position="center", max_lines=2, max_chars=15),
    "one_word_punch": _P("One-Word Punch", font_family="Anton", bold=False, animation="one_word", font_size=132, outline=8, position="center"),
    "word_reveal": _P("Word Reveal", font_family="Montserrat", animation="word_reveal", highlight_color="#FFFFFF", font_size=90, outline=5),
    "bebas_clean": _P("Bebas Clean", font_family="Bebas Neue", bold=False, font_size=110, outline=4, highlight_color="#FFFFFF", max_chars=22),
    "comic_bangers": _P("Comic Punch", font_family="Bangers", bold=False, primary_color="#FFE600", highlight_color="#FFFFFF", font_size=104, outline=6, max_chars=20),
    "slab_impact": _P("Slab Impact", font_family="Alfa Slab One", bold=False, animation="highlight", highlight_color="#FFD400", font_size=84, outline=6),
    "serif_elegant": _P("Serif Elegant", font_family="DM Serif Display", bold=False, uppercase=False, highlight_color="#FFD400", font_size=88, outline=2, shadow=3, max_chars=30),
    "boxed_tiktok": _P("Boxed", font_family="Roboto", background_enabled=True, background_color="#000000", highlight_color="#FFFFFF", font_size=78, outline=6, shadow=0, max_chars=24),
}

def get_preset(preset_id: str) -> dict:
    return STYLE_PRESETS.get(preset_id, STYLE_PRESETS["bold_white"])

print(f"🎭 Registered {len(STYLE_PRESETS)} creator caption presets.")


## 📝 Section 5: ASS Subtitle Generator


In [ ]:
# ==========================================
# 5. ASS SUBTITLE & CAPTION BUILDER
# ==========================================
def hex_to_ass(hex_color: str, alpha_hex: str = "00") -> str:
    """Convert #RRGGBB to ASS &HAABBGGRR color format."""
    c = hex_color.lstrip("#")
    if len(c) == 6:
        r, g, b = c[0:2], c[2:4], c[4:6]
        return f"&H{alpha_hex}{b}{g}{r}"
    return "&H00FFFFFF"

def format_ass_time(seconds: float) -> str:
    """Format seconds into ASS time format H:MM:SS.cs"""
    s = max(0.0, float(seconds))
    hrs = int(s // 3600)
    mins = int((s % 3600) // 60)
    secs = int(s % 60)
    cs = int(round((s - int(s)) * 100))
    if cs >= 100:
        secs += 1
        cs = 0
    return f"{hrs}:{mins:02d}:{secs:02d}.{cs:02d}"

def build_ass_subtitles(words: list[dict], preset_id: str, output_path: Path, frame_w: int = 1080, frame_h: int = 1920, src_w: int = 1920, src_h: int = 1080) -> Path:
    preset = get_preset(preset_id)
    
    font_name = preset.get("font_family", "Roboto")
    font_size = preset.get("font_size", 90)
    primary_color = hex_to_ass(preset.get("primary_color", "#FFFFFF"))
    highlight_color = hex_to_ass(preset.get("highlight_color", "#FFD400"))
    outline_color = hex_to_ass(preset.get("outline_color", "#000000"))
    back_color = hex_to_ass(preset.get("background_color", "#000000"), "80" if preset.get("background_enabled") else "00")
    
    bold = -1 if preset.get("bold", True) else 0
    outline = preset.get("outline", 5)
    shadow = preset.get("shadow", 1)
    
    # Calculate dynamic foreground layout
    src_ar = src_w / src_h if src_h > 0 else 16 / 9
    target_ar = frame_w / frame_h if frame_h > 0 else 9 / 16
    
    if src_ar > target_ar:
        fg_w = frame_w
        fg_h = int(round(frame_w / src_ar))
    else:
        fg_h = frame_h
        fg_w = int(round(frame_h * src_ar))
        
    fg_y = (frame_h - fg_h) // 2
    fg_bottom = fg_y + fg_h
    
    scaled_font_size = max(16, int(round(font_size * (frame_h / 1920.0))))
    caption_margin = int(round(frame_h * 0.02))
    caption_y = fg_bottom + caption_margin
    
    bottom_safe_margin = int(round(frame_h * 0.03))
    max_caption_bottom = frame_h - bottom_safe_margin
    max_lines = preset.get("max_lines", 2)
    caption_h_est = int(scaled_font_size * max_lines * 1.2) + outline * 2
    
    if caption_y + caption_h_est <= max_caption_bottom:
        alignment = 8
        margin_v = int(caption_y)
    elif (max_caption_bottom - fg_bottom) >= (scaled_font_size + caption_margin):
        alignment = 8
        margin_v = int(max(fg_bottom + 10, max_caption_bottom - caption_h_est))
    else:
        alignment = 2
        margin_v = int(frame_h * 0.10)
    
    header = f"""[Script Info]
ScriptType: v4.00+
PlayResX: {frame_w}
PlayResY: {frame_h}
ScaledBorderAndShadow: yes

[V4+ Styles]
Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding
Style: Default,{font_name},{scaled_font_size},{primary_color},{highlight_color},{outline_color},{back_color},{bold},0,0,0,100,100,0,0,1,{outline},{shadow},{alignment},20,20,{margin_v},1

[Events]
Format: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text
"""
    
    events = []
    if not words:
        output_path.write_text(header, encoding="utf-8")
        return output_path

    max_chars = preset.get("max_chars", 22)
    animation = preset.get("animation", "none")
    is_karaoke = preset.get("karaoke", False)
    uppercase = preset.get("uppercase", True)
    
    groups = []
    curr_group = []
    curr_len = 0
    
    for w in words:
        text = w["word"].upper() if uppercase else w["word"]
        w_item = {**w, "text": text}
        if curr_group and (curr_len + len(text) > max_chars):
            groups.append(curr_group)
            curr_group = [w_item]
            curr_len = len(text)
        else:
            curr_group.append(w_item)
            curr_len += len(text) + 1
    if curr_group:
        groups.append(curr_group)
        
    for grp in groups:
        start_t = format_ass_time(grp[0]["start"])
        end_t = format_ass_time(grp[-1]["end"])
        
        if is_karaoke or animation == "highlight":
            line_parts = []
            for item in grp:
                dur_cs = int(round((item["end"] - item["start"]) * 100))
                dur_cs = max(1, dur_cs)
                line_parts.append(f"{{\\k{dur_cs}}}{item['text']}")
            text_str = " ".join(line_parts)
        else:
            text_str = " ".join(item["text"] for item in grp)
            
        events.append(f"Dialogue: 0,{start_t},{end_t},Default,,0,0,0,,{text_str}")

    full_content = header + "\n".join(events)
    output_path.write_text(full_content, encoding="utf-8")
    return output_path


## ⬇️ Section 6: Production-Grade Video Downloader & Cookie System


In [ ]:
# ==========================================
# 6. PRODUCTION-GRADE YOUTUBE DOWNLOADER & COOKIE ENGINE
# ==========================================
import yt_dlp
import uuid
import time
import shutil
import logging
import json
import subprocess
import http.cookiejar
from pathlib import Path
from typing import Optional, Tuple, Dict, Any

logger = logging.getLogger("clipforge.downloader")
logger.setLevel(logging.INFO)

# --- NATIVE USER-AGENT MAPPING FOR YT-DLP CLIENT ROTATION ---
CLIENT_USER_AGENTS = {
    "web": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "tv": "Mozilla/5.0 (SmartTV; LINUX; Tizen 6.0) AppleWebKit/537.36 (KHTML, like Gecko) Version/6.0 TV Safari/537.36",
    "mweb": "Mozilla/5.0 (iPhone; CPU iPhone OS 17_4_1 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.4.1 Mobile/15E148 Safari/604.1",
    "embedded": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "ios": "com.google.ios.youtube/19.11.3 (iPhone15,2; U; CPU iOS 17_4_1 like Mac OS X; en_US)",
    "android": "com.google.android.youtube/19.11.38 (Linux; U; Android 14; en_US; Pixel 8 Build/UD1A.230803.041)",
    "music": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
}

# --- COOKIE VALIDATOR & PERSISTENCE MODULE ---
class CookieValidationResult:
    def __init__(self, is_valid: bool, message: str, total_cookies: int = 0, youtube_cookies: int = 0, expired_cookies: int = 0, has_auth: bool = False):
        self.is_valid = is_valid
        self.message = message
        self.total_cookies = total_cookies
        self.youtube_cookies = youtube_cookies
        self.expired_cookies = expired_cookies
        self.has_auth = has_auth

def prepare_and_validate_cookies(input_cookie_path: Optional[str]) -> Tuple[CookieValidationResult, Optional[Path]]:
    """
    Safely copies incoming cookies file to a persistent, writable location (PERSISTENT_COOKIE_FILE)
    and performs programmatic Netscape validation. Avoids using Gradio temporary upload paths directly.
    """
    target_cookie_path = PERSISTENT_COOKIE_FILE
    
    source_path = None
    if input_cookie_path and os.path.exists(input_cookie_path):
        source_path = Path(input_cookie_path)
    elif target_cookie_path.exists() and target_cookie_path.stat().st_size > 0:
        source_path = target_cookie_path
    elif (BASE_DIR / "drive" / "MyDrive" / "cookies.txt").exists():
        source_path = BASE_DIR / "drive" / "MyDrive" / "cookies.txt"
        
    if not source_path:
        return CookieValidationResult(False, "No cookies file provided."), None
        
    try:
        if source_path.resolve() != target_cookie_path.resolve():
            shutil.copy2(source_path, target_cookie_path)
            os.chmod(target_cookie_path, 0o666)
    except Exception as e:
        logger.warning(f"Failed to copy cookies to persistent path: {e}")
        target_cookie_path = source_path
        
    try:
        cj = http.cookiejar.MozillaCookieJar(str(target_cookie_path))
        cj.load(ignore_discard=True, ignore_expires=True)
    except Exception as e:
        return CookieValidationResult(False, f"Invalid Netscape cookies format: {e}"), None
        
    now = time.time()
    total_count = 0
    yt_count = 0
    expired_count = 0
    auth_found = False
    
    auth_cookie_names = {"SID", "__Secure-1PSID", "__Secure-3PSID", "LOGIN_INFO", "HSID", "SSID", "APISID", "SAPISID"}
    
    for cookie in cj:
        total_count += 1
        domain = cookie.domain or ""
        if "youtube.com" in domain or "google.com" in domain:
            yt_count += 1
            if cookie.name in auth_cookie_names:
                auth_found = True
            if cookie.expires and cookie.expires < now:
                expired_count += 1

    if yt_count == 0:
        return CookieValidationResult(False, "Cookies file contains no YouTube domain cookies.", total_count, 0, 0, False), target_cookie_path
        
    if expired_count >= yt_count and yt_count > 0:
        return CookieValidationResult(False, f"All YouTube cookies are expired ({expired_count}/{yt_count}).", total_count, yt_count, expired_count, auth_found), target_cookie_path
        
    msg = f"Valid cookies active ({yt_count} YouTube cookies, Auth session active: {auth_found})."
    return CookieValidationResult(True, msg, total_count, yt_count, expired_count, auth_found), target_cookie_path

# --- URL & PLATFORM VALIDATOR ---
def validate_url(url: str) -> Tuple[bool, str]:
    if not url or not isinstance(url, str):
        return False, "URL is empty or invalid."
    url_clean = url.strip()
    if not (url_clean.startswith("http://") or url_clean.startswith("https://")):
        return False, "URL must start with http:// or https://"
    return True, url_clean

# --- OPTIMIZED YT-DLP CLIENT ROTATION (HIGH-RES WEB/TV CLIENTS FIRST) --- 
CLIENT_ROTATION = [
    ("web", {"extractor_args": {"youtube": {"player_client": ["web"]}}}),
    ("tv", {"extractor_args": {"youtube": {"player_client": ["tv"]}}}),
    ("mweb", {"extractor_args": {"youtube": {"player_client": ["mweb"]}}}),
    ("embedded", {"extractor_args": {"youtube": {"player_client": ["embedded"]}}}),
    ("ios", {"extractor_args": {"youtube": {"player_client": ["ios"]}}}),
    ("android", {"extractor_args": {"youtube": {"player_client": ["android"]}}}),
    ("music", {"extractor_args": {"youtube": {"player_client": ["music"]}}}),
]

# --- UNCONSTRAINED HIGH-QUALITY FORMAT CASCADE (AV1 -> VP9 -> AVC) ---
FORMAT_FALLBACK_CASCADE = [
    "bestvideo[vcodec^=av01]+bestaudio/bestvideo[vcodec^=vp09]+bestaudio/bestvideo[vcodec^=vp9]+bestaudio/bestvideo[vcodec^=avc1]+bestaudio/bestvideo+bestaudio/best",
    "bv*+ba/b",
    "bestvideo+bestaudio",
    "best",
]

def is_bot_or_auth_error(reason: str) -> bool:
    r = (reason or "").lower()
    return any(k in r for k in (
        "sign in to confirm you're not a bot", "confirm you're not a bot",
        "sign in", "login", "bot", "cookie", "members-only", "private video",
        "age-gated", "account"
    ))

# --- AUTOMATIC PRE-DOWNLOAD FORMAT INSPECTOR ---
def inspect_available_formats(url: str, active_cookie_path: Optional[Path] = None) -> Dict[str, Any]:
    """Inspect all available formats for a video before downloading to determine maximum quality targets."""
    opts = {
        'quiet': True,
        'no_warnings': True,
        'nocheckcertificate': True,
    }
    if active_cookie_path:
        opts['cookiefile'] = str(active_cookie_path)

    try:
        with yt_dlp.YoutubeDL(opts) as ydl:
            info = ydl.extract_info(url, download=False)
            formats = info.get('formats', [])
            title = info.get('title', 'Unknown Title')
            
            max_height = 0
            max_fps = 0
            max_v_bitrate = 0
            max_a_bitrate = 0
            hdr_supported = False
            codecs_found = set()
            audio_codecs_found = set()
            
            v_formats = []
            a_formats = []
            
            for f in formats:
                vcodec = f.get('vcodec', 'none')
                acodec = f.get('acodec', 'none')
                height = f.get('height') or 0
                fps = f.get('fps') or 0
                vbr = f.get('vbr') or f.get('tbr') or 0
                abr = f.get('abr') or 0

                if f.get('dynamic_range') in ['HDR', 'HDR10', 'HLG', 'PQ']:
                    hdr_supported = True

                if vcodec != 'none':
                    max_height = max(max_height, height)
                    max_fps = max(max_fps, fps)
                    max_v_bitrate = max(max_v_bitrate, vbr)
                    if 'av01' in vcodec:
                        codecs_found.add('AV1')
                    elif 'vp09' in vcodec or 'vp9' in vcodec:
                        codecs_found.add('VP9')
                    elif 'avc' in vcodec or 'h264' in vcodec:
                        codecs_found.add('AVC')
                    v_formats.append(f)

                if acodec != 'none':
                    max_a_bitrate = max(max_a_bitrate, abr)
                    if 'opus' in acodec:
                        audio_codecs_found.add('Opus')
                    elif 'mp4a' in acodec or 'aac' in acodec:
                        audio_codecs_found.add('AAC')
                    a_formats.append(f)

            print("==================================================")
            print(f"📊 [PRE-DOWNLOAD AUDIT - AVAILABLE FORMATS]: {title}")
            print(f"   • Available Video Formats Count: {len(v_formats)}")
            print(f"   • Available Audio Formats Count: {len(a_formats)}")
            print(f"   • Maximum Resolution Available: {max_height}p")
            print(f"   • Maximum Frame Rate Available: {max_fps} fps")
            print(f"   • Maximum Video Bitrate: {max_v_bitrate:.0f} kbps")
            print(f"   • Maximum Audio Bitrate: {max_a_bitrate:.0f} kbps")
            print(f"   • Video Codecs Available: {', '.join(sorted(codecs_found)) or 'Unknown'}")
            print(f"   • Audio Codecs Available: {', '.join(sorted(audio_codecs_found)) or 'Unknown'}")
            print(f"   • HDR Support: {'Yes 🌈' if hdr_supported else 'No (SDR)'}")
            print("==================================================")

            return {
                'title': title,
                'max_height': max_height,
                'max_fps': max_fps,
                'max_v_bitrate': max_v_bitrate,
                'max_a_bitrate': max_a_bitrate,
                'hdr_supported': hdr_supported,
                'codecs': codecs_found,
                'audio_codecs': audio_codecs_found,
                'formats': formats
            }
    except Exception as e:
        print(f"⚠️ Pre-download inspection warning: {e}")
        return {'title': 'Video', 'max_height': 0, 'max_fps': 0, 'max_v_bitrate': 0, 'max_a_bitrate': 0, 'hdr_supported': False, 'codecs': set(), 'audio_codecs': set(), 'formats': []}

# --- POST-DOWNLOAD METADATA VALIDATION & AUDIT ---
def audit_downloaded_file(
    file_path: Path, 
    selected_client: str, 
    selected_fmt_id: str, 
    selected_extractor: str,
    max_audit: Dict[str, Any]
) -> Dict[str, Any]:
    """Run ffprobe on downloaded file and audit whether maximum quality was achieved."""
    cmd = [
        "ffprobe", "-v", "quiet",
        "-print_format", "json",
        "-show_format", "-show_streams",
        str(file_path)
    ]
    res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    if res.returncode != 0:
        print(f"⚠️ ffprobe post-download audit warning: {res.stderr}")
        return {}

    probe = json.loads(res.stdout)
    streams = probe.get("streams", [])
    fmt_info = probe.get("format", {})

    v_stream = next((s for s in streams if s.get("codec_type") == "video"), {})
    a_stream = next((s for s in streams if s.get("codec_type") == "audio"), {})

    w = int(v_stream.get("width", 0))
    h = int(v_stream.get("height", 0))
    vcodec = v_stream.get("codec_name", "N/A")
    acodec = a_stream.get("codec_name", "N/A")
    fps_eval = v_stream.get("r_frame_rate", "0/1")
    try:
        num, den = map(float, fps_eval.split("/"))
        fps = round(num / den, 2) if den > 0 else 0
    except Exception:
        fps = 0

    bitrate_bps = int(v_stream.get("bit_rate") or fmt_info.get("bit_rate") or 0)
    bitrate_kbps = bitrate_bps // 1000
    duration = float(fmt_info.get("duration", 0))

    print("\n==================================================")
    print(f"📥 [POST-DOWNLOAD VALIDATION & AUDIT REPORT]")
    print(f"   • File Path: {file_path.name}")
    print(f"   • Resolution: {w}x{h} (Height: {h}p)")
    print(f"   • Codec: Video ({vcodec}) | Audio ({acodec})")
    print(f"   • FPS: {fps}")
    print(f"   • Bitrate: {bitrate_kbps} kbps")
    print(f"   • Duration: {duration:.2f} seconds")
    print(f"   • Video Stream Details: {v_stream.get('pix_fmt', 'yuv420p')} | {v_stream.get('color_transfer', 'SDR')}")
    print(f"   • Audio Stream Details: {a_stream.get('sample_rate', '48000')} Hz | {a_stream.get('channels', 2)} ch")
    print(f"   • Selected Format ID: {selected_fmt_id}")
    print(f"   • Selected Client: {selected_client}")
    print(f"   • Selected Extractor: {selected_extractor}")
    print(f"   • File Size: {file_path.stat().st_size / (1024*1024):.2f} MB")
    print("==================================================")

    # --- QUALITY DEGRADATION WARNING SYSTEM ---
    max_h = max_audit.get('max_height', 0)
    warnings = []
    if max_h > 0 and h < max_h:
        warnings.append(f"Resolution degraded: Downloaded {h}p but maximum available format is {max_h}p.")
    if max_audit.get('max_fps', 0) > 0 and fps < (max_audit.get('max_fps') - 2):
        warnings.append(f"FPS degraded: Downloaded {fps} fps but maximum available format is {max_audit.get('max_fps')} fps.")
    if 'AV1' in max_audit.get('codecs', set()) and 'av01' not in vcodec and 'vp09' not in vcodec:
        warnings.append(f"Codec fallback: Downloaded legacy {vcodec} codec while high-efficiency AV1/VP9 was available.")

    if warnings:
        print("⚠️ [QUALITY DEGRADATION WARNING]:")
        for w_msg in warnings:
            print(f"   ❌ {w_msg}")
        print("   💡 Note: If YouTube restricted high resolutions for unauthenticated client sessions, provide a cookies.txt file to unlock 4K/8K formats.")
        print("==================================================\n")
    else:
        print("🎉 [PERFECT QUALITY]: Downloaded highest available resolution & quality manifest!\n==================================================\n")

    return {
        'resolution': f"{w}x{h}",
        'codec': f"{vcodec}/{acodec}",
        'fps': fps,
        'bitrate': f"{bitrate_kbps} kbps",
        'duration': duration,
        'video_stream': v_stream,
        'audio_stream': a_stream,
        'format_id': selected_fmt_id,
        'client': selected_client,
        'extractor': selected_extractor
    }

def download_video_url(
    url: str,
    output_dir: Path = DOWNLOADS_DIR,
    cookies_file: Optional[str] = None,
    max_retries: int = 2
) -> Tuple[Path, str]:
    """Production-grade video downloader with pre-download format inspection, AV1/VP9 priority, and post-download audit."""
    is_valid_url, clean_url = validate_url(url)
    if not is_valid_url:
        raise ValueError(clean_url)

    # 1. Prepare persistent cookies & validate
    cookie_result, active_cookie_path = prepare_and_validate_cookies(cookies_file)
    
    if cookies_file and not cookie_result.is_valid:
        print(f"⚠️ Cookie Warning: {cookie_result.message}. Proceeding unauthenticated...")
    elif active_cookie_path and cookie_result.is_valid:
        print(f"🍪 {cookie_result.message}")

    # 2. Inspect available formats before downloading
    max_audit = inspect_available_formats(clean_url, active_cookie_path)

    try:
        ytdlp_ver = yt_dlp.version.__version__
    except Exception:
        ytdlp_ver = "latest"

    file_id = uuid.uuid4().hex
    out_template = str(output_dir / f"{file_id}.%(ext)s")

    # 3. Base Options with Network Resiliency & 10MB Chunks
    base_opts = {
        'outtmpl': out_template,
        'quiet': True,
        'no_warnings': True,
        'retries': 5,
        'fragment_retries': 5,
        'ignoreerrors': False,
        'nocheckcertificate': True,
        'socket_timeout': 30,
        'http_chunk_size': 10485760,  # 10MB chunking prevents bandwidth throttling
        'concurrent_fragment_downloads': 4,
        'merge_output_format': 'mkv',  # Lossless container merge without codec rejection
    }

    if active_cookie_path and cookie_result.is_valid:
        base_opts['cookiefile'] = str(active_cookie_path)

    last_exception = None
    bot_detected = False

    for client_name, client_args in CLIENT_ROTATION:
        ua = CLIENT_USER_AGENTS.get(client_name, CLIENT_USER_AGENTS["web"])
        
        headers_opts = {
            'http_headers': {
                'User-Agent': ua,
                'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
                'Accept-Language': 'en-US,en;q=0.9',
                'Sec-Fetch-Mode': 'navigate',
            }
        }

        for fmt in FORMAT_FALLBACK_CASCADE:
            opts = {**base_opts, **client_args, **headers_opts, 'format': fmt}

            try:
                print(f"🔍 [DIAGNOSTICS] yt-dlp v{ytdlp_ver} | Client: '{client_name}' | Format: '{fmt[:40]}...'")

                with yt_dlp.YoutubeDL(opts) as ydl:
                    info = ydl.extract_info(clean_url, download=True)
                    title = info.get("title", "Video")
                    fmt_id = info.get("format_id", "Unknown")
                    extractor = info.get("extractor_key", "YouTube")

                    matches = list(output_dir.glob(f"{file_id}.*"))
                    if matches and matches[0].stat().st_size > 0:
                        selected_file = matches[0]
                        
                        # Post-download validation & metadata printout
                        audit_downloaded_file(
                            file_path=selected_file,
                            selected_client=client_name,
                            selected_fmt_id=fmt_id,
                            selected_extractor=extractor,
                            max_audit=max_audit
                        )
                        return selected_file, title

            except Exception as e:
                err_str = str(e)
                logger.warning(f"Failed client='{client_name}', format='{fmt[:30]}': {err_str[:200]}")
                last_exception = e

                if is_bot_or_auth_error(err_str):
                    bot_detected = True
                    break

                time.sleep(0.5)

        if bot_detected:
            break

    if bot_detected and (not active_cookie_path or not cookie_result.is_valid):
        error_msg = (
            "🔒 **YouTube Bot / Sign-in Verification Required**\n\n"
            "YouTube is blocking unauthenticated downloads for this video.\n\n"
            "**How to resolve:**\n"
            "1. Export your YouTube cookies file (`cookies.txt`) using a browser extension (e.g., *Get cookies.txt LOCALLY*).\n"
            "2. Upload the `cookies.txt` file in the **🍪 Optional: Upload cookies.txt** box in the Gradio settings.\n"
            "3. Click **Generate Shorts** again."
        )
        raise RuntimeError(error_msg)
    else:
        error_msg = f"❌ Download failed after format & client fallbacks. Technical details: {last_exception}"
        raise RuntimeError(error_msg)


## 🎙️ Section 7: Whisper Transcriber & Hinglish Romanisation


In [ ]:
# ==========================================
# 7. WHISPER TRANSCRIBER & HINGLISH TRANSLITERATOR
# ==========================================
from faster_whisper import WhisperModel

_cached_models: dict[str, WhisperModel] = {}

def get_whisper_model(model_size: str = "medium", device: str = "auto") -> tuple[WhisperModel, str]:
    target_device = "cuda" if (device in ("auto", "cuda") and cuda_available) else "cpu"
    compute_type = "float16" if target_device == "cuda" else "int8"
    cache_key = f"{model_size}_{target_device}_{compute_type}"
    
    if cache_key in _cached_models:
        return _cached_models[cache_key], target_device
        
    print(f"⚡ Loading Whisper model '{model_size}' on {target_device.upper()} ({compute_type})...")
    model = WhisperModel(model_size, device=target_device, compute_type=compute_type)
    _cached_models[cache_key] = model
    return model, target_device

def transcribe_video(video_path: Path, model_size: str = "medium", device: str = "auto") -> dict:
    model, used_device = get_whisper_model(model_size, device)
    
    print(f"🎙️ Transcribing {video_path.name} with word-level timestamps...")
    segments, info = model.transcribe(
        str(video_path),
        beam_size=5,
        word_timestamps=True,
        vad_filter=True,
    )
    
    segment_list = []
    all_words = []
    
    for seg in segments:
        seg_words = []
        if seg.words:
            for w in seg.words:
                w_dict = {"word": w.word.strip(), "start": w.start, "end": w.end, "probability": w.probability}
                seg_words.append(w_dict)
                all_words.append(w_dict)
                
        segment_list.append({
            "id": seg.id,
            "start": seg.start,
            "end": seg.end,
            "text": seg.text.strip(),
            "words": seg_words
        })
        
    return {
        "language": info.language,
        "language_probability": info.language_probability,
        "duration": info.duration,
        "segments": segment_list,
        "words": all_words,
        "device": used_device,
    }


## 🧠 Section 8: Intelligent Local Clip Selector


In [ ]:
# ==========================================
# 8. LOCAL INTELLIGENT CLIP SELECTOR
# ==========================================
STRONG_WORDS = {
    "how", "why", "what", "when", "who", "where", "best", "worst", "never",
    "always", "secret", "mistake", "biggest", "important", "actually", "truth",
    "realize", "realise", "amazing", "incredible", "stop", "avoid", "must",
    "everyone", "nobody", "money", "free", "new", "first", "tip", "tips",
}

def select_clips(transcript: dict, num_clips: int = 3, target_duration: float = 30.0) -> list[dict]:
    segments = transcript.get("segments") or []
    if not segments:
        duration = transcript.get("duration", 60.0)
        step = duration / (num_clips + 1)
        return [{"start": i * step, "end": min(duration, i * step + target_duration), "title": f"Clip {i+1}"} for i in range(num_clips)]
    
    total_duration = transcript.get("duration", segments[-1]["end"])
    
    candidates = []
    for i, start_seg in enumerate(segments):
        start_t = start_seg["start"]
        end_t = start_t + target_duration
        if end_t > total_duration:
            end_t = total_duration
            
        if end_t - start_t < 10.0:
            continue
            
        window_text = []
        word_count = 0
        hook_score = 0
        
        for seg in segments:
            if seg["end"] >= start_t and seg["start"] <= end_t:
                window_text.append(seg["text"])
                for w in seg.get("words", []):
                    word_count += 1
                    if w["word"].lower().strip(".,!?") in STRONG_WORDS:
                        hook_score += 1.5
                        
        combined_text = " ".join(window_text)
        density = word_count / max(1.0, (end_t - start_t))
        score = (hook_score * 2.0) + (density * 1.5)
        
        title = combined_text[:40].strip() + "..." if len(combined_text) > 40 else combined_text
        if not title:
            title = f"Highlight {len(candidates)+1}"
            
        candidates.append({
            "start": start_t,
            "end": end_t,
            "score": score,
            "title": title
        })
        
    candidates.sort(key=lambda x: x["score"], reverse=True)
    
    selected = []
    for cand in candidates:
        overlap = False
        for sel in selected:
            if not (cand["end"] <= sel["start"] or cand["start"] >= sel["end"]):
                overlap = True
                break
        if not overlap:
            selected.append(cand)
            if len(selected) >= num_clips:
                break
                
    selected.sort(key=lambda x: x["start"])
    return selected


## 🎬 Section 9: FFmpeg Video Clipper & Reframer


In [ ]:
# ==========================================
# 9. FFMPEG REFRAMING & SUBTITLE BURNING (MAX QUALITY ENGINE)
# ==========================================
import json
import subprocess
from pathlib import Path

def probe_media_streams(media_path: Path) -> dict:
    """Run ffprobe on media file to inspect resolution, bitrate, fps, and codecs."""
    cmd = [
        "ffprobe", "-v", "quiet",
        "-print_format", "json",
        "-show_format", "-show_streams",
        str(media_path)
    ]
    res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    if res.returncode != 0:
        raise RuntimeError(f"ffprobe failed to inspect file: {res.stderr}")
    return json.loads(res.stdout)

def render_clip(
    source_video: Path,
    start_time: float,
    end_time: float,
    output_path: Path,
    ass_path: Path,
    aspect_ratio: AspectRatio = AspectRatio.NINE_16,
    fit_mode: FitMode = FitMode.CROP,
    bar_text: str = None
) -> Path:
    """High-fidelity FFmpeg clip renderer maintaining maximum source quality (CRF 18, Lanczos scaling, 320k audio)."""
    output_path.parent.mkdir(parents=True, exist_ok=True)
    clip_dur = max(0.1, float(end_time - start_time))

    # 1. Inspect source stream specs with ffprobe
    probe_data = probe_media_streams(source_video)
    streams = probe_data.get("streams", [])
    format_info = probe_data.get("format", {})
    
    video_stream = next((s for s in streams if s.get("codec_type") == "video"), None)
    audio_stream = next((s for s in streams if s.get("codec_type") == "audio"), None)

    if not video_stream:
        err_msg = f"❌ Cannot render clip: Source media file '{source_video.name}' contains NO video stream. Stopping render."
        print(err_msg)
        raise RuntimeError(err_msg)

    # Extract source quality metrics
    src_w = int(video_stream.get("width", 1080))
    src_h = int(video_stream.get("height", 1920))
    src_vcodec = video_stream.get("codec_name", "h264")
    src_acodec = audio_stream.get("codec_name", "aac") if audio_stream else "N/A"
    pix_fmt = video_stream.get("pix_fmt", "yuv420p")
    color_transfer = video_stream.get("color_transfer", "bt709")
    
    src_bitrate_bps = int(video_stream.get("bit_rate") or format_info.get("bit_rate") or 0)
    src_bitrate_kbps = f"{src_bitrate_bps // 1000} kbps" if src_bitrate_bps > 0 else "Dynamic/Variable"
    src_fps = video_stream.get("r_frame_rate", "30/1")

    print("==================================================")
    print(f"🎬 [QUALITY INSPECTION - RENDER SOURCE]: {source_video.name}")
    print(f"   • Source Resolution: {src_w}x{src_h}")
    print(f"   • Source Bitrate: {src_bitrate_kbps}")
    print(f"   • Source Codec: Video ({src_vcodec}) | Audio ({src_acodec})")
    print(f"   • Pixel Format: {pix_fmt} | Transfer: {color_transfer}")
    print(f"   • Source Frame Rate: {src_fps} fps")

    # 2. Target Canvas Dimensions (1080x1920 for 9:16)
    if aspect_ratio == AspectRatio.NINE_16:
        target_w = 1080
        target_h = 1920
    else:
        target_w = 1920
        target_h = 1080

    # Calculate exact uncropped foreground dimensions matching original aspect ratio
    src_ar = src_w / src_h if src_h > 0 else (16 / 9)
    target_ar = target_w / target_h

    if src_ar > target_ar:
        fg_w = target_w
        fg_h = int(round(target_w / src_ar))
    else:
        fg_h = target_h
        fg_w = int(round(target_h * src_ar))

    fg_w = fg_w if fg_w % 2 == 0 else fg_w - 1
    fg_h = fg_h if fg_h % 2 == 0 else fg_h - 1

    fg_x = (target_w - fg_w) // 2
    fg_y = (target_h - fg_h) // 2

    # Prepare escaped path for ASS subtitle filter in FFmpeg
    clean_ass_path = str(ass_path.resolve()).replace("\\", "/")
    clean_fonts_dir = str(FONTS_DIR.resolve()).replace("\\", "/")
    clean_ass_path = clean_ass_path.replace(":", "\\:")
    clean_fonts_dir = clean_fonts_dir.replace(":", "\\:")

    # 3. High-Quality 2-Layer Filtergraph: Blurred Background + Sharp Uncropped Foreground
    filtergraph = (
        f"[0:v]scale={target_w}:{target_h}:force_original_aspect_ratio=increase:flags=bicubic,"
        f"crop={target_w}:{target_h},"
        f"gblur=sigma=30:steps=3,"
        f"eq=brightness=-0.15:contrast=0.95[bg];"
        f"[0:v]scale={fg_w}:{fg_h}:flags=lanczos[fg];"
        f"[bg][fg]overlay={fg_x}:{fg_y}[vid_base];"
        f"[vid_base]ass='{clean_ass_path}':fontsdir='{clean_fonts_dir}'[outv]"
    )

    # 4. Visually Lossless Encoding Settings (CRF 18, 320k Audio, Lanczos)
    stream_mapping = ["[outv]", "0:a?"]
    cmd = [
        "ffmpeg", "-y",
        "-ss", f"{start_time:.3f}",
        "-to", f"{end_time:.3f}",
        "-i", str(source_video),
        "-filter_complex", filtergraph,
        "-map", stream_mapping[0],
        "-map", stream_mapping[1],
        "-c:v", "libx264",
        "-preset", "medium",
        "-crf", "18",
        "-pix_fmt", "yuv420p",
        "-colorspace", "bt709",
        "-color_primaries", "bt709",
        "-color_trc", "bt709",
        "-c:a", "aac",
        "-b:a", "320k",
        str(output_path)
    ]

    cmd_str = " ".join(cmd)
    print(f"⚡ [FFMPEG COMMAND]:\n   {cmd_str}")

    # 5. Execute FFmpeg
    res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    if res.returncode != 0 or not output_path.exists() or output_path.stat().st_size == 0:
        raise RuntimeError(f"FFmpeg render failed:\n{res.stderr[-800:]}")

    # 6. Verify Output Quality Metrics
    out_probe = probe_media_streams(output_path)
    out_v_stream = next((s for s in out_probe.get("streams", []) if s.get("codec_type") == "video"), {})
    out_a_stream = next((s for s in out_probe.get("streams", []) if s.get("codec_type") == "audio"), {})
    out_fmt = out_probe.get("format", {})

    out_w = out_v_stream.get("width", 0)
    out_h = out_v_stream.get("height", 0)
    out_vcodec = out_v_stream.get("codec_name", "h264")
    out_acodec = out_a_stream.get("codec_name", "aac")
    
    out_bitrate_bps = int(out_v_stream.get("bit_rate") or out_fmt.get("bit_rate") or 0)
    out_bitrate_kbps = f"{out_bitrate_bps // 1000} kbps" if out_bitrate_bps > 0 else "Dynamic (CRF 18 High Quality)"

    print(f"🎬 [QUALITY INSPECTION - RENDER OUTPUT]: {output_path.name}")
    print(f"   • Output Resolution: {out_w}x{out_h}")
    print(f"   • Output Bitrate: {out_bitrate_kbps}")
    print(f"   • Output Codec: Video ({out_vcodec}) | Audio ({out_acodec})")
    print(f"   • Output File Size: {output_path.stat().st_size / (1024*1024):.2f} MB")
    print("==================================================")

    return output_path


## ⚡ Section 10: Complete Pipeline Orchestrator & ZIP Exporter


In [ ]:
# ==========================================
# 10. MAIN PIPELINE & ZIP EXPORTER
# ==========================================
import zipfile
import shutil

def process_video_pipeline(
    video_url: str = None,
    uploaded_file: str = None,
    cookies_file: str = None,
    num_clips: int = 3,
    aspect_ratio: str = "9:16",
    fit_mode: str = "crop",
    caption_style: str = "bold_white",
    compute_device: str = "auto",
    whisper_model: str = "medium",
    clip_duration: float = 30.0,
    save_to_drive: bool = False,
    progress_callback = None
) -> tuple[list[str], str, str]:
    """Execute complete end-to-end shorts generation pipeline."""
    
    def log(msg, pct):
        print(f"[{int(pct*100)}%] {msg}")
        if progress_callback:
            progress_callback(pct, msg)

    job_id = uuid.uuid4().hex[:8]
    job_clip_dir = CLIPS_DIR / job_id
    job_clip_dir.mkdir(parents=True, exist_ok=True)
    
    # 1. Obtain video source file
    log("📥 Downloading / processing source video...", 0.05)
    if uploaded_file and os.path.exists(uploaded_file):
        source_path = Path(uploaded_file)
        source_title = source_path.stem
    elif video_url and video_url.strip():
        source_path, source_title = download_video_url(
            url=video_url.strip(), 
            output_dir=DOWNLOADS_DIR, 
            cookies_file=cookies_file
        )
    else:
        raise ValueError("Please provide either a valid YouTube URL or upload a video file.")

    # 2. Transcribe
    log("🎙️ Transcribing audio with Whisper...", 0.25)
    transcript = transcribe_video(source_path, model_size=whisper_model, device=compute_device)
    
    # 3. Select Clips
    log("🧠 Selecting best viral clip windows...", 0.60)
    selected_clips = select_clips(transcript, num_clips=num_clips, target_duration=clip_duration)
    
    rendered_video_paths = []
    
    # 4. Render each clip
    total_clips = len(selected_clips)
    for idx, clip in enumerate(selected_clips):
        pct = 0.70 + (0.25 * (idx / max(1, total_clips)))
        log(f"✂️ Rendering short clip {idx+1}/{total_clips}: '{clip['title']}'...", pct)
        
        clip_words = [
            {"word": w["word"], "start": w["start"] - clip["start"], "end": w["end"] - clip["start"]}
            for w in transcript["words"]
            if w["start"] >= clip["start"] and w["end"] <= clip["end"]
        ]
        
        ass_path = job_clip_dir / f"sub_{idx}.ass"
        ar_enum = AspectRatio.NINE_16 if aspect_ratio == "9:16" else AspectRatio.SIXTEEN_9
        fit_enum = FitMode.SQUARE if fit_mode == "square" else FitMode.CROP
        
        # Probe source video dimensions for caption layout positioning
        try:
            probe_data = probe_media_streams(source_path)
            v_stream = next((s for s in probe_data.get("streams", []) if s.get("codec_type") == "video"), None)
            src_w = int(v_stream.get("width", 1920)) if v_stream else 1920
            src_h = int(v_stream.get("height", 1080)) if v_stream else 1080
        except Exception:
            src_w, src_h = 1920, 1080

        w_dim, h_dim = (1080, 1920) if ar_enum == AspectRatio.NINE_16 or fit_enum == FitMode.SQUARE else (1920, 1080)
        build_ass_subtitles(clip_words, caption_style, ass_path, frame_w=w_dim, frame_h=h_dim, src_w=src_w, src_h=src_h)
        
        out_path = job_clip_dir / f"clip_{idx+1}.mp4"
        render_clip(
            source_video=source_path,
            start_time=clip["start"],
            end_time=clip["end"],
            output_path=out_path,
            ass_path=ass_path,
            aspect_ratio=ar_enum,
            fit_mode=fit_enum
        )
        rendered_video_paths.append(str(out_path))

    # 5. Create ZIP Archive
    log("📦 Packaging clips into ZIP download...", 0.95)
    zip_path = EXPORT_DIR / f"ClipForge_Shorts_{job_id}.zip"
    with zipfile.ZipFile(zip_path, 'w') as zipf:
        for vp in rendered_video_paths:
            p = Path(vp)
            zipf.write(p, arcname=p.name)
            
    drive_msg = ""
    if save_to_drive:
        gdrive_dir = mount_gdrive()
        if gdrive_dir:
            dest_job = gdrive_dir / f"Shorts_{job_id}"
            dest_job.mkdir(parents=True, exist_ok=True)
            for vp in rendered_video_paths:
                shutil.copy(vp, dest_job / Path(vp).name)
            shutil.copy(zip_path, dest_job / zip_path.name)
            drive_msg = f" (Saved to Google Drive: {dest_job})"

    log(f"🎉 Complete! Generated {len(rendered_video_paths)} shorts successfully.{drive_msg}", 1.0)
    
    status_summary = f"### ✅ Successfully generated {len(rendered_video_paths)} clips!\n- **Source**: {source_title}\n- **Whisper Model**: {whisper_model} ({transcript['device'].upper()})\n- **Caption Style**: {caption_style}\n{drive_msg}"
    
    return rendered_video_paths, str(zip_path), status_summary


## 🔍 Section 11: System Verification & Diagnostics


In [ ]:
# ==========================================
# 11. PRE-FLIGHT VERIFICATION
# ==========================================
print("🔍 Running system diagnostics...")
assert DOWNLOADS_DIR.exists(), "Downloads directory missing"
assert CLIPS_DIR.exists(), "Clips directory missing"
assert (FONTS_DIR / "Roboto-Regular.ttf").exists(), "Roboto font missing"
assert (MASKS_DIR / "rounded_1020_60.png").exists(), "Rounded mask missing"

ff_check = subprocess.run(["ffmpeg", "-version"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
assert ff_check.returncode == 0, "FFmpeg is not accessible"

print(f"✅ All system checks passed!")
print(f"   • FFmpeg: OK")
print(f"   • CUDA/GPU: {'Available (' + gpu_name + ')' if cuda_available else 'CPU Fallback'}")
print(f"   • Fonts & Assets: Ready")


## 🖥️ Section 12: Launch Gradio Web Interface


In [ ]:
# ==========================================
# 12. GRADIO WEB INTERFACE
# ==========================================
import gradio as gr

def create_ui():
    style_choices = list(STYLE_PRESETS.keys())
    
    with gr.Blocks(title="ClipForge — Local AI Video Clipper") as demo:
        gr.Markdown(
            """
            # 🎬 ClipForge — Local AI Video Clipper
            ### Turn any YouTube video or upload into short, reframed, styled-captioned clips!
            """
        )
        
        with gr.Row():
            with gr.Column(scale=1):
                gr.Markdown("### 📥 1. Source Video")
                video_url_input = gr.Textbox(
                    label="YouTube URL",
                    placeholder="https://www.youtube.com/watch?v=...",
                    lines=1
                )
                video_file_input = gr.Video(
                    label="Or Upload Video File",
                    sources=["upload"]
                )
                cookies_file_input = gr.File(
                    label="🍪 Optional: Upload cookies.txt (Bypass YouTube bot verification / sign-in requirement)",
                    file_types=[".txt"],
                    file_count="single"
                )
                
                gr.Markdown("### ⚙️ 2. Generation Settings")
                with gr.Row():
                    num_clips_input = gr.Slider(minimum=1, maximum=10, value=3, step=1, label="Number of Clips")
                    clip_duration_input = gr.Slider(minimum=15, maximum=60, value=30, step=5, label="Clip Duration (s)")
                
                with gr.Row():
                    aspect_ratio_input = gr.Dropdown(choices=["9:16", "16:9"], value="9:16", label="Aspect Ratio")
                    fit_mode_input = gr.Dropdown(choices=["crop", "square"], value="crop", label="Fit Mode")
                
                with gr.Row():
                    caption_style_input = gr.Dropdown(choices=style_choices, value="bold_white", label="Caption Style")
                    whisper_model_input = gr.Dropdown(choices=["tiny", "base", "small", "medium", "large-v3"], value="medium", label="Whisper Model")
                
                with gr.Row():
                    compute_device_input = gr.Dropdown(choices=["auto", "cuda", "cpu"], value="auto", label="Compute Device")
                    save_drive_input = gr.Checkbox(label="Save to Google Drive", value=False)
                
                with gr.Row():
                    generate_btn = gr.Button("🚀 Generate Shorts", variant="primary", size="lg")
                    clear_btn = gr.Button("🧹 Clear Inputs", size="lg")
            
            with gr.Column(scale=1):
                gr.Markdown("### 📺 3. Generated Shorts & Download")
                status_output = gr.Markdown(value="*Click 'Generate Shorts' to begin processing...*")
                
                gallery_output = gr.Gallery(
                    label="Preview Shorts",
                    show_label=True,
                    elem_id="gallery",
                    columns=2,
                    height="auto"
                )
                
                zip_download_output = gr.File(label="📦 Download All Clips (ZIP Archive)")

        def on_generate(url, file, cookies, num_clips, aspect, fit, style, device, model, duration, drive, progress=gr.Progress()):
            def update_progress(pct, msg):
                progress(pct, desc=msg)
                
            cookies_path = cookies.name if cookies is not None else None
            try:
                clips, zip_path, summary = process_video_pipeline(
                    video_url=url,
                    uploaded_file=file,
                    cookies_file=cookies_path,
                    num_clips=int(num_clips),
                    aspect_ratio=aspect,
                    fit_mode=fit,
                    caption_style=style,
                    compute_device=device,
                    whisper_model=model,
                    clip_duration=float(duration),
                    save_to_drive=drive,
                    progress_callback=update_progress
                )
                return clips, zip_path, summary
            except Exception as e:
                err_msg = f"{str(e)}"
                return [], None, err_msg

        def on_clear():
            return "", None, None, "*Inputs cleared.*", [], None

        generate_btn.click(
            fn=on_generate,
            inputs=[
                video_url_input, video_file_input, cookies_file_input, num_clips_input,
                aspect_ratio_input, fit_mode_input, caption_style_input,
                compute_device_input, whisper_model_input, clip_duration_input,
                save_drive_input
            ],
            outputs=[gallery_output, zip_download_output, status_output]
        )
        
        clear_btn.click(
            fn=on_clear,
            inputs=[],
            outputs=[video_url_input, video_file_input, cookies_file_input, status_output, gallery_output, zip_download_output]
        )

    return demo

demo = create_ui()
demo.queue()
demo.launch(share=True, debug=True, inline=True)
